# Linear Transformations

| Difficulty | ███░░░░░░░ 3/10 |
| :--- | :--- |
| Prerequisites | 01.02 |
| Time | ~65 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/01_Linear_Algebra/03_linear_transformations.ipynb)

---

## 🎯 Learning Objective

See linear maps geometrically — rotation, scaling, shearing — and learn how changing basis changes the coordinates of a transformation without changing the transformation itself.

---

## 📐 Theory

A function $T$ is a **linear transformation** if it respects addition and scaling:

$$T(a\mathbf{u} + b\mathbf{v}) = aT(\mathbf{u}) + bT(\mathbf{v})$$

Every linear transformation on $\mathbb{R}^n$ can be represented by a matrix, and applying the
transformation is just matrix-vector multiplication: $T(\mathbf{x}) = A\mathbf{x}$.

### The standard building blocks

$$R(\theta) = \begin{pmatrix}\cos\theta & -\sin\theta \\ \sin\theta & \cos\theta\end{pmatrix}
\qquad
S(s_x, s_y) = \begin{pmatrix}s_x & 0 \\ 0 & s_y\end{pmatrix}
\qquad
\text{Shear} = \begin{pmatrix}1 & k \\ 0 & 1\end{pmatrix}$$

Rotation preserves lengths and angles; scaling stretches/compresses along the axes
independently; shear slides one axis proportionally to the other, distorting angles while
preserving area.

### Composition

Applying $T_1$ then $T_2$ is itself linear, and its matrix is the **product** $A_2 A_1$ (note
the order: the transformation applied *first* goes on the *right*). Because matrix
multiplication is not commutative in general, **order matters** — rotating then scaling is
usually not the same as scaling then rotating.

### Change of basis

A vector's coordinates depend on which basis you describe it in. If $B = [\mathbf{b}_1 \mid
\mathbf{b}_2 \mid \cdots]$ collects a new basis's vectors as columns, then a vector with
coordinates $\mathbf{c}$ in that basis has standard coordinates $\mathbf{x} = B\mathbf{c}$, so
converting standard coordinates *into* the new basis is $\mathbf{c} = B^{-1}\mathbf{x}$. The
transformation itself doesn't change — only the numbers describing it do.

---

## 👁️ Visual Intuition

Watching a grid of points get rotated, scaled, and sheared makes each transformation's
geometric signature obvious: rotation spins the grid rigidly, scaling stretches it
axis-by-axis, and shear tilts it like a deck of cards.

In [ ]:
# Rotation, scaling, and shear applied to a grid of points
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

def rotation_matrix(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s], [s, c]])

def scaling_matrix(sx, sy):
    return np.array([[sx, 0.0], [0.0, sy]])

def shear_matrix(k):
    return np.array([[1.0, k], [0.0, 1.0]])

xs, ys = np.meshgrid(np.linspace(-1, 1, 9), np.linspace(-1, 1, 9))
grid = np.stack([xs.ravel(), ys.ravel()])

transforms = {
    "Rotation (45deg)": rotation_matrix(np.pi / 4),
    "Scaling (2x, 0.5x)": scaling_matrix(2.0, 0.5),
    "Shear (k=0.8)": shear_matrix(0.8),
}

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
for ax, (title, A) in zip(axes, transforms.items()):
    original = grid
    transformed = A @ grid
    ax.scatter(*original, s=6, color="lightgray", label="original")
    ax.scatter(*transformed, s=10, color="#4C72B0", label="transformed")
    ax.set_title(title)
    ax.set_aspect('equal')
    ax.set_xlim(-2.5, 2.5); ax.set_ylim(-2.5, 2.5)
    ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

## 🐍 Implementation from Scratch

Composition of transformations is matrix multiplication — but order matters. We verify this
directly by applying two transformations sequentially versus multiplying their matrices in
each possible order.

In [ ]:
# Composition = matrix multiplication, and order changes the result
import numpy as np

def rotation_matrix(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s], [s, c]])

def scaling_matrix(sx, sy):
    return np.array([[sx, 0.0], [0.0, sy]])

R = rotation_matrix(np.pi / 3)
Sc = scaling_matrix(2.0, 0.5)
x = np.array([1.0, 0.0])

# "Scale, then rotate" means apply Sc first, then R -- matrices compose as R @ Sc
scale_then_rotate = R @ (Sc @ x)
combined_1 = (R @ Sc) @ x
print("Scale-then-rotate matches (R @ Sc) @ x:", np.allclose(scale_then_rotate, combined_1))

# "Rotate, then scale" is the OPPOSITE order: Sc @ R
rotate_then_scale = Sc @ (R @ x)
combined_2 = (Sc @ R) @ x
print("Rotate-then-scale matches (Sc @ R) @ x:", np.allclose(rotate_then_scale, combined_2))

print("\nAre the two orders the same result?", np.allclose(scale_then_rotate, rotate_then_scale))
print("scale-then-rotate:", np.round(scale_then_rotate, 4))
print("rotate-then-scale:", np.round(rotate_then_scale, 4))
print("Matrix multiplication is NOT commutative in general -- order of operations matters.")

## 🤖 Why This Matters for AI

Every layer of a neural network is (at minimum) a linear transformation followed by a
nonlinearity. Stacking layers stacks these warps of space. Below, we push a synthetic
two-class dataset that is **not** linearly separable through two random (untrained) linear +
ReLU layers, purely to see the geometric *mechanism* at work. With **trained** weights, this
warping becomes dramatic enough to make the classes linearly separable by the final layer —
this exact picture is why deep networks can solve problems a single linear classifier cannot.

In [ ]:
# Watching a linear map + nonlinearity warp a 2D point cloud (untrained weights -- illustrative)
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons

X, y = make_moons(n_samples=200, noise=0.08, random_state=0)

rng = np.random.default_rng(1)
W1 = rng.standard_normal((2, 2))
b1 = rng.standard_normal(2) * 0.1
W2 = rng.standard_normal((2, 2))
b2 = rng.standard_normal(2) * 0.1

def relu(z):
    return np.maximum(0.0, z)

H = relu(X @ W1.T + b1)     # layer 1: linear map + ReLU nonlinearity
Z = H @ W2.T + b2            # layer 2: another linear map

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, data, title in zip(axes, [X, H, Z],
                            ["Input space\n(not linearly separable)", "After layer 1\n(linear + ReLU)", "After layer 2"]):
    ax.scatter(data[:, 0], data[:, 1], c=y, cmap="coolwarm", s=15, edgecolor='k', linewidth=0.2)
    ax.set_title(title, fontsize=10)
plt.tight_layout()
plt.show()
print("With RANDOM weights the classes are still tangled -- training adjusts W1, W2, b1, b2")
print("specifically so that the final space becomes linearly separable by a simple boundary.")

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. Compute $R(60°)$ by hand from
   $R(\theta)=\begin{pmatrix}\cos\theta&-\sin\theta\\\sin\theta&\cos\theta\end{pmatrix}$, then
   verify against `rotation_matrix(np.pi/3)`. As a structural sanity check, confirm
   $R(60°)^\top R(60°)=I$ and $\det(R(60°))=1$ — both must hold for *any* rotation matrix, at
   *any* angle — then confirm the same two checks hold for $R(117°)$.

<details>
<summary>💡 Solution</summary>

$\cos60°=0.5$, $\sin60°=\frac{\sqrt3}{2}\approx0.8660$, so
$$R(60°)=\begin{pmatrix}0.5&-0.8660\\0.8660&0.5\end{pmatrix},$$
matching `rotation_matrix(np.pi/3)` to floating-point precision. $R(60°)^\top R(60°)=I$
(orthogonal — its columns are unit length and perpendicular) and
$\det(R(60°))=0.5^2+0.8660^2=1$ exactly. Both checks also hold for $R(117°)$:
$\det=\cos^2117°+\sin^2117°=1$ always, by the Pythagorean identity, regardless of the angle —
rotations never scale or reflect, so both invariants hold identically for every $\theta$.

</details>

### 💭 UNDERSTAND
2. Without computing anything yet: for the vector $\mathbf{x}=(1,0)$, do you expect "scale by
   $(3,1)$, then rotate $90°$" and "rotate $90°$, then scale by $(3,1)$" to land on the same
   point? Compute both (using `rotation_matrix` and `scaling_matrix`) to check, then explain —
   in terms of *which axis gets stretched* — why non-uniform scaling and rotation don't commute,
   while uniform scaling ($s_x=s_y$) and rotation always do.

<details>
<summary>💡 Solution</summary>

They disagree: scale-then-rotate sends $(1,0)\to(3,0)\to(0,3)$, while rotate-then-scale sends
$(1,0)\to(0,1)\to(0,1)$ (scaling by $(3,1)$ leaves a vector already on the $y$-axis unchanged,
since $s_y=1$). Non-uniform scaling stretches *specific, fixed* coordinate axes — after a $90°$
rotation, "the $x$-axis" is a completely different direction in the plane than it was before, so
scaling the *rotated* frame's axes by $(3,1)$ stretches different geometric directions than
scaling the *original* frame would have. Uniform scaling ($s_x=s_y=s$) has no such asymmetry —
it's just "multiply every vector's length by $s$," a property of the vector alone, not of which
axis it happens to line up with — so it doesn't matter whether you scale before or after
rotating.

</details>

### ✏️ DERIVE
3. If $\mathbf{c}=B^{-1}\mathbf{x}$ are $\mathbf{x}$'s coordinates in a new basis $B$, derive
   that the *same* linear map $\mathbf{y}=A\mathbf{x}$, re-expressed in that basis, is
   $\mathbf{c}_y = A'\mathbf{c}$ where $A'=B^{-1}AB$ — i.e. $A'$ is $A$'s matrix "as seen from"
   the $B$-basis. Start from $\mathbf{c}_y = B^{-1}\mathbf{y}$.

<details>
<summary>💡 Solution outline</summary>

$\mathbf{y}$'s coordinates in the $B$-basis are, by the same rule used to get $\mathbf{c}$,
$\mathbf{c}_y=B^{-1}\mathbf{y}$. Substitute $\mathbf{y}=A\mathbf{x}$:
$$\mathbf{c}_y = B^{-1}(A\mathbf{x}).$$
Now substitute $\mathbf{x}=B\mathbf{c}$ (the inverse of $\mathbf{c}=B^{-1}\mathbf{x}$):
$$\mathbf{c}_y = B^{-1}A(B\mathbf{c}) = (B^{-1}AB)\mathbf{c} = A'\mathbf{c}.$$
So the single matrix $A'=B^{-1}AB$ takes $B$-basis coordinates *in* directly to $B$-basis
coordinates *out*, without ever converting back to standard coordinates in between. Numeric
check with $B=\begin{pmatrix}2&1\\1&1\end{pmatrix}$, $A=R(90°)$, $\mathbf{x}=(5,3)$:
$\mathbf{c}=B^{-1}\mathbf{x}=(2,1)$, $A'=B^{-1}AB=\begin{pmatrix}-3&-2\\5&3\end{pmatrix}$, and
$A'\mathbf{c}=(-8,13)$ matches $B^{-1}(A\mathbf{x})=(-8,13)$ computed the "long way" exactly.

</details>

### 🐍 IMPLEMENT
4. Write `compose_transforms(*matrices)` that returns the single matrix equivalent to applying
   `matrices[0]` first, then `matrices[1]`, and so on (matching the notebook's own
   composition-order convention). Test it on a shear, a scaling, and a rotation matrix of your
   choice.

<details>
<summary>✅ How to know you're right</summary>

Applying the matrices sequentially to a test vector (`M3 @ (M2 @ (M1 @ x))`) must match applying
`compose_transforms(M1, M2, M3) @ x` exactly, for several different test vectors `x` — since
matrix multiplication is associative, any discrepancy means a bug in your composition order, not
a real mathematical ambiguity. As an independent structural check, confirm
$\det(\text{compose\_transforms}(M_1,M_2,M_3)) = \det(M_1)\det(M_2)\det(M_3)$ — determinants are
multiplicative under composition for *any* square matrices, so this must hold regardless of
which three transforms you picked.

</details>

### 🤖 APPLY
5. Two attention heads describe the same token vector $\mathbf{x}_{\text{tok}}$ in two different
   subspace bases, $B_1$ and $B_2$ (invertible $2\times2$ matrices, standing in for each head's
   learned projection). Compute $\mathbf{x}_{\text{tok}}$'s coordinates in each basis,
   $\mathbf{c}_1=B_1^{-1}\mathbf{x}_{\text{tok}}$ and $\mathbf{c}_2=B_2^{-1}\mathbf{x}_{\text{tok}}$,
   and confirm both reconstruct $\mathbf{x}_{\text{tok}}$ exactly via $B_1\mathbf{c}_1$ and
   $B_2\mathbf{c}_2$.

<details>
<summary>✅ What you should observe</summary>

$\mathbf{c}_1$ and $\mathbf{c}_2$ come out *different* from each other (for example, with
$B_1=\begin{pmatrix}1&1\\1&-1\end{pmatrix}$, $B_2=\begin{pmatrix}0&1\\2&1\end{pmatrix}$,
$\mathbf{x}_{\text{tok}}=(2,1)$: $\mathbf{c}_1=(1.5,0.5)$ but $\mathbf{c}_2=(-0.5,2.0)$) — yet
$B_1\mathbf{c}_1$ and $B_2\mathbf{c}_2$ both reconstruct $(2,1)$ exactly. This is the
linear-algebra fact underneath multi-head attention's per-head projections: the *same*
underlying token is validly described by entirely different coordinate vectors depending on
which head's (basis's) point of view you ask from, and no information is lost in either
direction — exactly the round-trip this notebook's own change-of-basis formula guarantees for
*any* invertible $B$, not something specific to attention.

</details>

### 🚀 CHALLENGE
6. The notebook's own "Why This Matters for AI" cell pushes `make_moons` data through 2
   *random* (untrained) linear+ReLU layers, both with a 2-to-2 (no bottleneck) shape.
   Investigate what happens with *more* random layers, same 2-to-2 shape throughout: measure how
   well a linear classifier can separate the two classes after 1, 2, 4, and 8 such layers (e.g.
   via `LogisticRegression().fit(H, y).score(H, y)`, averaged over several random seeds per
   depth). Does stacking more random, untrained, narrow layers make the classes easier or harder
   to linearly separate — and why might that be, given what each individual layer does to the
   *space*?

<details>
<summary>🔍 What a strong answer covers</summary>

A strong answer runs this and finds separability *decreases* with depth on average, for this
fixed narrow (2-dimensional) width: roughly 0.87 accuracy on the raw untransformed data, dropping
to roughly 0.78 after 1 layer, 0.73 after 2, 0.68 after 4, and 0.54 (barely better than the 0.50
of random guessing) after 8 — averaged over many random seeds, with substantial seed-to-seed
variance at every depth. A strong answer connects this to the mechanism, not just the trend: each
*untrained* linear+ReLU layer is a random, not-yet-useful warp of a space that's already only 2D
— with no bottleneck to force useful compression and no training signal to shape the warp
usefully, composing more of these random maps just keeps re-mixing the same limited 2D
information, and ReLU's zeroing-out of negative entries destroys a little more of it each time it
fires. This is the same phenomenon the notebook's own 2-layer demo already shows qualitatively
(random weights leave the classes tangled), pushed further: depth alone, without training, is
not what makes deep networks powerful — the *learned* weights are, and this exercise's random
baseline is exactly the "before training" half of that story.

</details>

---

## 📚 Further Reading
- Strang, *Introduction to Linear Algebra*, Ch. 2 (Solving Linear Equations) & Ch. 6 (change of basis)
- [3Blue1Brown: Linear transformations and matrices](https://www.3blue1brown.com/lessons/linear-transformations)

---

*Next notebook: [Systems of Linear Equations](04_systems_of_equations.ipynb)*